# Практическая работа 7. Задание 2
A* в лабиринте 8x8: манхэттенская и евклидова эвристики, сравнение с Дейкстрой

In [1]:
import heapq
import math

maze = [
    'S..#....',
    '.#.#.##.',
    '.#....#.',
    '.####.#.',
    '....#...',
    '###.###.',
    '........',
    '.#####.G'
]
grid = [list(row) for row in maze]
start = (0, 0)
goal = (7, 7)


def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def euclid(a, b):
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def astar_grid(grid, start, goal, heuristic):
    n, m = len(grid), len(grid[0])
    open_set = [(heuristic(start, goal), start)]
    g_score = {start: 0}
    came_from = {}
    closed = set()
    explored = 0  # считаем клетки в момент извлечения из очереди
    while open_set:
        _, current = heapq.heappop(open_set)
        if current in closed:
            continue
        closed.add(current)
        explored += 1
        if current == goal:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return path[::-1], g_score[goal], explored
        r, c = current
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < n and 0 <= nc < m):
                continue
            if grid[nr][nc] == '#':
                continue
            neighbor = (nr, nc)
            tentative_g = g_score[current] + 1
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f, neighbor))
    return None, float('inf'), explored


def dijkstra_grid(grid, start, goal):
    n, m = len(grid), len(grid[0])
    visited = set()
    dist = {start: 0}
    open_set = [(0, start)]
    while open_set:
        d, current = heapq.heappop(open_set)
        if current in visited:
            continue
        visited.add(current)
        if current == goal:
            return dist[goal], len(visited)
        r, c = current
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < n and 0 <= nc < m):
                continue
            if grid[nr][nc] == '#':
                continue
            neighbor = (nr, nc)
            new_d = d + 1
            if neighbor not in dist or new_d < dist[neighbor]:
                dist[neighbor] = new_d
                heapq.heappush(open_set, (new_d, neighbor))
    return None, len(visited)


def show(grid, path):
    # рисуем путь звездочками
    g = [row[:] for row in grid]
    for r, c in path[1:-1]:
        g[r][c] = '*'
    for row in g:
        print(' '.join(row))

In [2]:
path_m, cost_m, expl_m = astar_grid(grid, start, goal, manhattan)
print('A* (манхэттен): длина', cost_m, ', исследовано клеток', expl_m)
print('Путь (строка, столбец):')
print(' '.join('(%d,%d)' % p for p in path_m))
show(grid, path_m)

A* (манхэттен): длина 14 , исследовано клеток 28
Путь (строка, столбец):
(0,0) (0,1) (0,2) (1,2) (2,2) (2,3) (2,4) (2,5) (3,5) (4,5) (4,6) (4,7) (5,7) (6,7) (7,7)
S * * # . . . .
. # * # . # # .
. # * * * * # .
. # # # # * # .
. . . . # * * *
# # # . # # # *
. . . . . . . *
. # # # # # . G


In [3]:
path_e, cost_e, expl_e = astar_grid(grid, start, goal, euclid)
print('A* (евклид): длина', cost_e, ', исследовано клеток', expl_e)
print('Путь (строка, столбец):')
print(' '.join('(%d,%d)' % p for p in path_e))
show(grid, path_e)
print('Пути совпадают:', path_m == path_e)

cost_d, expl_d = dijkstra_grid(grid, start, goal)
print()
print('Дейкстра: длина', cost_d, ', исследовано клеток', expl_d)
free = sum(row.count('.') for row in maze) + 2
print('Всего свободных клеток:', free)

A* (евклид): длина 14 , исследовано клеток 29
Путь (строка, столбец):
(0,0) (1,0) (2,0) (3,0) (4,0) (4,1) (4,2) (4,3) (5,3) (6,3) (6,4) (6,5) (6,6) (6,7) (7,7)
S . . # . . . .
* # . # . # # .
* # . . . . # .
* # # # # . # .
* * * * # . . .
# # # * # # # .
. . . * * * * *
. # # # # # . G
Пути совпадают: False

Дейкстра: длина 14 , исследовано клеток 40
Всего свободных клеток: 40
